In [ ]:
#!/usr/bin/env python
"""
CLIF 2.1 -> 3.0 migration.

For a site's data folder, this script:
  * crosswalks every BETA table (converts only the standardized
    *_category / *_group / *_type VALUES; nothing else is touched),
  * copies non-beta CLIF tables through UNCHANGED so the 3.0 output is complete,
  * never reads or writes junk / PHI files,
  * verifies each conversion preserved row count, column set, and distinct ID
    counts (schema/metadata only — no full data load),
  * reports timezone changes (the DuckDB backend relabels tz-aware timestamps to
    UTC; the instants are preserved, so that relabel is NOT treated as an error),
  * wraps each table in try/except so one bad table never aborts the whole run,
  * logs everything to the console and to a timestamped log file, and prints
    where the converted files and the log are stored.

Assumes parquet input (the verification uses parquet metadata).
Requires the clifpy build that ships crosswalk_file_2_1_to_3_0 / BETA_TABLES.
"""

import logging
import shutil
import traceback
from datetime import datetime
from pathlib import Path

import duckdb
import pyarrow as pa
import pyarrow.parquet as pq

from clifpy import ClifOrchestrator, crosswalk_file_2_1_to_3_0, BETA_TABLES

# --------------------------------------------------------------------------- #
# Config
# --------------------------------------------------------------------------- #
CONFIG_PATH = "../config/demo_data_config.yaml"
ID_COLS = ["patient_id", "hospitalization_id"]
EXCLUDE = []


co = ClifOrchestrator(config_path=CONFIG_PATH)
data_dir = Path(co.data_directory)
out_dir = Path(co.output_directory)
out_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------- #
# Logging: console + a timestamped log file in the output directory
# --------------------------------------------------------------------------- #
log_path = out_dir / f"crosswalk_2.1_to_3.0_{datetime.now():%Y%m%d_%H%M%S}.log"
log = logging.getLogger("clif_crosswalk")
log.setLevel(logging.INFO)
log.handlers.clear()           # avoid duplicate handlers if re-run in a notebook
log.propagate = False
_fmt = logging.Formatter("%(asctime)s  %(levelname)-7s  %(message)s")
for _h in (logging.FileHandler(log_path, encoding="utf-8"), logging.StreamHandler()):
    _h.setFormatter(_fmt)
    log.addHandler(_h)

# --------------------------------------------------------------------------- #
# Helpers (schema / metadata only — no full data load)
# --------------------------------------------------------------------------- #
def tz_map(path):
    """{column: tz} for timestamp columns. tz=None means timezone-naive."""
    return {f.name: f.type.tz
            for f in pq.read_schema(path)
            if pa.types.is_timestamp(f.type)}


def zones(tzmap):
    """Compact set of timezones across timestamp columns ('naive' = no tz)."""
    return ",".join(sorted({(v or "naive") for v in tzmap.values()})) or "-"


def summary(path):
    """rows, column set, distinct ID counts, and per-column tz — no data load."""
    schema = pq.read_schema(path)
    cols = schema.names
    s = {
        "rows": pq.read_metadata(path).num_rows,
        "cols": set(cols),
        "tz": {f.name: f.type.tz for f in schema if pa.types.is_timestamp(f.type)},
    }
    for c in ID_COLS:
        if c in cols:
            s[c] = duckdb.sql(
                f"SELECT COUNT(DISTINCT {c}) FROM read_parquet('{path.as_posix()}')"
            ).fetchone()[0]
    return s


def tz_status(src_tz, dst_tz):
    """Return (description, is_concern).

    A plain relabel of a tz-aware column to UTC is what the DuckDB backend does
    and is instant-preserving, so it is NOT flagged as a concern. Anything else
    (a real zone shift, or tz-aware -> naive) IS a concern worth investigating.
    """
    if src_tz == dst_tz:
        return "match", False
    diffs = {c: (src_tz.get(c), dst_tz.get(c))
             for c in set(src_tz) | set(dst_tz)
             if src_tz.get(c) != dst_tz.get(c)}
    relabel_only = all(new == "UTC" and old is not None for old, new in diffs.values())
    if relabel_only:
        return "relabel->UTC (instants preserved)", False
    return f"CHANGED {diffs}", True


# --------------------------------------------------------------------------- #
# Pre-flight: account for EVERY file in the folder
# --------------------------------------------------------------------------- #
ALL = {p.stem.removeprefix("clif_"): p
       for p in sorted(data_dir.glob(f"*.{co.filetype}"))}

beta     = [t for t in ALL if t in BETA_TABLES]
other    = [t for t in ALL if t not in BETA_TABLES and t not in EXCLUDE]
excluded = [t for t in ALL if t in EXCLUDE]
missing  = [t for t in BETA_TABLES if t not in ALL]

log.info("CLIF 2.1 -> 3.0 migration starting")
log.info("Data dir : %s", data_dir.resolve())
log.info("Output   : %s", out_dir.resolve())
log.info("Crosswalk (beta tables present)      : %s", beta)
log.info("Copy-through (other CLIF, not in scope): %s", other)
log.info("Beta tables MISSING from this folder   : %s", missing)
if not ALL:
    log.warning("No '*.%s' files found in %s — nothing to do.", co.filetype, data_dir)
log.info("-" * 90)

# --------------------------------------------------------------------------- #
# Run
# --------------------------------------------------------------------------- #
results = {}
n_converted = n_mismatch = n_copied = n_skipped = n_failed = n_incomplete = 0

# 1) Crosswalk the beta tables -------------------------------------------------
for tb in beta:
    in_path = ALL[tb]
    out_path = out_dir / in_path.name

    if out_path.exists():
        log.info("%-32s SKIP (output already exists)", tb)
        n_skipped += 1
        continue

    try:
        report = crosswalk_file_2_1_to_3_0(str(in_path), str(out_path), tb)

        # Verify nothing but the sanctioned values changed.
        src, dst = summary(in_path), summary(out_path)
        checks = {"rows": src["rows"] == dst["rows"],
                  "cols": src["cols"] == dst["cols"]}
        for c in ID_COLS:
            if c in src:
                checks[c] = src[c] == dst[c]
        integrity_ok = all(checks.values())

        tz_desc, tz_concern = tz_status(src["tz"], dst["tz"])

        if not integrity_ok:
            verdict = "*** MISMATCH ***"
        elif tz_concern:
            verdict = "TZ-WARN"
        else:
            verdict = "OK"

        results[tb] = {"is_complete": report["is_complete"],
                       "checks": checks, "tz": tz_desc, "report": report}

        ids = "  ".join(f"{c}:{src[c]}->{dst[c]}" for c in ID_COLS if c in src)
        log.info("%-32s complete=%-5s %-16s rows %s->%s  tz %s->%s  %s",
                 tb, report["is_complete"], verdict,
                 src["rows"], dst["rows"],
                 zones(src["tz"]), zones(dst["tz"]), ids)

        n_converted += 1
        if not integrity_ok:
            n_mismatch += 1
            log.error("   %s INTEGRITY FAILED -> %s",
                      tb, [k for k, v in checks.items() if not v])
        if tz_concern:
            log.warning("   %s timezone change: %s", tb, tz_desc)
        if not report["is_complete"]:
            n_incomplete += 1

    except Exception:
        n_failed += 1
        log.error("%-32s FAILED to convert\n%s", tb, traceback.format_exc())
        # Remove any partial/corrupt output so a rerun retries this table cleanly.
        if out_path.exists():
            try:
                out_path.unlink()
            except OSError:
                log.error("   could not remove partial output %s", out_path)
        continue

# 2) Copy non-beta CLIF tables through UNCHANGED ------------------------------
for tb in other:
    in_path = ALL[tb]
    out_path = out_dir / in_path.name
    if out_path.exists():
        log.info("%-32s SKIP copy (output already exists)", tb)
        n_skipped += 1
        continue
    try:
        shutil.copy2(in_path, out_path)        # byte-for-byte; changes nothing
        n_copied += 1
        log.info("%-32s COPIED unchanged (not in crosswalk scope)", tb)
    except Exception:
        n_failed += 1
        log.error("%-32s FAILED to copy\n%s", tb, traceback.format_exc())

# --------------------------------------------------------------------------- #
# Tables needing manual mapping (is_complete = False)
# --------------------------------------------------------------------------- #
incomplete = {tb: r for tb, r in results.items() if not r["is_complete"]}
if incomplete:
    log.info("-" * 90)
    log.info("VALUES NEEDING MANUAL MAPPING (is_complete=False) — left as-is in output:")
    for tb, r in incomplete.items():
        for col, info in r["report"].get("columns", {}).items():
            flagged = (info.get("ambiguous") or []) + (info.get("unresolved") or [])
            if flagged:
                vals = [f.get("original") for f in flagged]
                log.info("   %-28s %-24s %s", tb, col, vals)

# --------------------------------------------------------------------------- #
# Summary + where it's stored
# --------------------------------------------------------------------------- #
log.info("=" * 90)
log.info("DONE.  converted=%d  copied=%d  skipped=%d  failed=%d  mismatch=%d  needs-review=%d",
         n_converted, n_copied, n_skipped, n_failed, n_mismatch, n_incomplete)
log.info("CLIF 3.0 output written to: %s", out_dir.resolve())
log.info("Run log saved to:           %s", log_path.resolve())

print()
print(f"CLIF 3.0 files are in: {out_dir.resolve()}")
print(f"Run log:               {log_path.resolve()}")
if n_failed or n_mismatch:
    print(f"Attention: failed={n_failed}, mismatch={n_mismatch} -> see the log above.")
if n_incomplete:
    print(f"{n_incomplete} table(s) have values needing manual mapping (is_complete=False).")